### Solar PV Only and Solar PV + BESS

In [1]:
import os
import sys
import csv

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
current_dir = os.getcwd()
project_root = current_dir
while project_root != os.path.dirname(project_root):
    if os.path.exists(os.path.join(project_root, "enpax")):
        break
    project_root = os.path.dirname(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

from enpax.runner import CentralRunner

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
SYSTEM_SIZE_KWDC = 198_200
ILR              = 1.17
BESS_MW          = [15, 30, 45, 60, 75, 90, 105, 120, 135, 150]
BESS_DURATIONS   = [2, 4, 8, 12]
SYSTEM_SIZE_MWDC = SYSTEM_SIZE_KWDC / 1_000

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def extract_capex_items(tech):
    return {
        cat: data
        for cat, data in tech.capex.line_items.items()
        if data["total"] > 0
    }

def extract_opex_items(tech):
    return {
        comp: val
        for comp, val in tech.opex.line_items.items()
        if val > 0
    }

# ---------------------------------------------------------------------------
# Run standalone solar
# ---------------------------------------------------------------------------


standalone_cfg = {
    "project_name": "Standalone Solar",
    "technologies": [{
        "name": "standalone_solar",
        "type": "solar_bess_2024Q1",
        "params": {
            "SystemSize": SYSTEM_SIZE_KWDC,
            "ILR":        ILR,
            "IncludeESS": False,
            "Cost_Office_Interconnect": 0,
            "Office_InterconnectFixed": 0,
            "Cost_EBOS_NetwUpgrade":  16_000_000 / SYSTEM_SIZE_KWDC * ILR,
            "SalesTaxRate": 0,
            
        },
    }],
}

standalone_tech = CentralRunner(standalone_cfg).run().tech_results[0]

scenarios = []

# Column 0 — standalone
scenarios.append({
    "solar_mwdc": SYSTEM_SIZE_MWDC,
    "bess_mw":    0,
    "duration_h": 0,
    "capex_kw":   standalone_tech.capex.total,
    "capex_items": extract_capex_items(standalone_tech),
    "opex_kwyr":  standalone_tech.opex.annual_total,
    "opex_items": extract_opex_items(standalone_tech),
})

# ---------------------------------------------------------------------------
# Run BESS sweep
# ---------------------------------------------------------------------------
for bess_mw in BESS_MW:
    for dur in BESS_DURATIONS:
        storage_duration = bess_mw * dur / SYSTEM_SIZE_MWDC

        cfg = {
            "project_name": f"Solar+BESS {bess_mw:.0f}MW {dur}h",
            "technologies": [{
                "name": f"solar_bess_{bess_mw:.0f}mw_{dur}h",
                "type": "solar_bess_2024Q1",
                "params": {
                    "SystemSize":      SYSTEM_SIZE_KWDC,
                    "ILR":             ILR,
                    "StorageDuration": storage_duration,
                    "BatteryDuration": dur,
                    "IncludeESS":      True,
                    "Cost_Office_Interconnect": 0,
                    "Office_InterconnectFixed": 0,
                    "Cost_EBOS_NetwUpgrade":  16_000_000 / SYSTEM_SIZE_KWDC * ILR,
                    "SalesTaxRate": 0,
                },
            }],
        }

        tech = CentralRunner(cfg).run().tech_results[0]

        #print(tech)

        scenarios.append({
            "solar_mwdc": SYSTEM_SIZE_MWDC,
            "bess_mw":    bess_mw,
            "duration_h": dur,
            "capex_kw":   tech.capex.total,
            "capex_items": extract_capex_items(tech),
            "opex_kwyr":  tech.opex.annual_total,
            "opex_items": extract_opex_items(tech),
        })

# ---------------------------------------------------------------------------
# Build CAPEX keys
# ---------------------------------------------------------------------------
capex_seen = {}
for sc in scenarios:
    for cat, data in sc["capex_items"].items():
        key_cat = (cat, "")
        capex_seen.setdefault(key_cat, None)
        for comp, val in data["components"].items():
            if val > 0:
                capex_seen.setdefault((cat, comp), None)

capex_keys = list(capex_seen.keys())

# ---------------------------------------------------------------------------
# Build OPEX keys
# ---------------------------------------------------------------------------
opex_seen = {}
for sc in scenarios:
    for comp in sc["opex_items"]:
        opex_seen.setdefault(comp, None)

opex_keys = list(opex_seen.keys())

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def capex_val(sc, cat, comp):
    items = sc["capex_items"]
    if cat not in items:
        return 0.0
    if comp == "":
        return items[cat]["total"]
    return items[cat]["components"].get(comp, 0.0)

def opex_val(sc, comp):
    return sc["opex_items"].get(comp, 0.0)

# ---------------------------------------------------------------------------
# Build rows
# ---------------------------------------------------------------------------
n = len(scenarios)
rows = []

# Metadata
rows.append(["Solar Capacity", "MWdc"] + [f"{sc['solar_mwdc']:.1f}" for sc in scenarios])
rows.append(["BESS Capacity", "MW"] + [f"{sc['bess_mw']:.0f}" for sc in scenarios])
rows.append(["BESS Duration", "h"] + [str(sc["duration_h"]) for sc in scenarios])

# ---------------- CAPEX ----------------
rows.append(["CAPEX", "$/kWdc"] + [""] * n)

all_cats = list(dict.fromkeys(cat for cat, _ in capex_keys))

for cat in all_cats:
    rows.append(
        [cat, "$/kWdc"] +
        [f"{capex_val(sc, cat, ''):.4f}" for sc in scenarios]
    )

    comps = [(c, comp) for c, comp in capex_keys if c == cat and comp != ""]
    for _, comp in comps:
        rows.append(
            [f"  {comp}", "$/kWdc"] +
            [f"{capex_val(sc, cat, comp):.4f}" for sc in scenarios]
        )

rows.append(
    ["TOTAL CAPEX", "$/kWdc"] +
    [f"{sc['capex_kw']:.4f}" for sc in scenarios]
)

# spacer
rows.append(["", ""] + [""] * n)

# ---------------- OPEX ----------------
rows.append(["OPEX", "$/kWdc-yr"] + [""] * n)

for comp in opex_keys:
    rows.append(
        [comp, "$/kWdc-yr"] +
        [f"{opex_val(sc, comp):.4f}" for sc in scenarios]
    )

rows.append(
    ["TOTAL OPEX", "$/kWdc-yr"] +
    [f"{sc['opex_kwyr']:.4f}" for sc in scenarios]
)

# ---------------------------------------------------------------------------
# Write CSV
# ---------------------------------------------------------------------------
OUTPUT_CSV = "solar_bess_capex_opex_sweep.csv"

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Metric", "Unit"] + [f"Scenario_{i}" for i in range(n)])
    writer.writerows(rows)

print(f"CSV saved → {OUTPUT_CSV}")
print(f"Scenarios: {n} | CAPEX rows: {len(capex_keys)} | OPEX rows: {len(opex_keys)}")

CSV saved → solar_bess_capex_opex_sweep.csv
Scenarios: 41 | CAPEX rows: 73 | OPEX rows: 10


### BESS Only

In [2]:
import os
import sys
import csv

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
current_dir = os.getcwd()
project_root = current_dir
while project_root != os.path.dirname(project_root):
    if os.path.exists(os.path.join(project_root, "enpax")):
        break
    project_root = os.path.dirname(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

from enpax.runner import CentralRunner

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
BESS_MW        = [15, 30, 45, 60, 75, 90, 105, 120, 135, 150]
BESS_DURATIONS = [2, 4, 8, 12]

INTERCONNECT_FIXED = 16_000_000
INTERCONNECT_VAR   = 0

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def extract_capex_items(tech):
    return dict(tech.capex.line_items)

def extract_opex_items(tech):
    return dict(tech.opex.line_items)

# ---------------------------------------------------------------------------
# Run BESS sweep
# ---------------------------------------------------------------------------
scenarios = []

for bess_mw in BESS_MW:
    for dur in BESS_DURATIONS:
        bess_mwac = bess_mw
        network_upgrade_per_kwac = 16_000_000 / (bess_mwac * 1_000)

        cfg = {
            "project_name": f"Standalone BESS {bess_mw:.0f}MW {dur}h",
            "technologies": [{
                "name": f"standalone_bess_{bess_mw:.0f}mw_{dur}h",
                "type": "bess_2025",
                "params": {
                    "BatteryCapacity": bess_mw,
                    "BatteryDuration": dur,
                    "NetworkUpgrade":    0,
                    "InterconnectFixed": 0,
                    "Interconnect":      0,
                },
            }],
        }

        tech = CentralRunner(cfg).run().tech_results[0]

        #print(tech)

        scenarios.append({
            "bess_mw":    bess_mw,
            "duration_h": dur,
            "bess_mwh":   bess_mw * dur,
            "capex_kwh":  tech.capex.total,
            "capex_items": extract_capex_items(tech),
            "opex_kwh":   tech.opex.annual_total,
            "opex_items": extract_opex_items(tech),
        })

# ---------------------------------------------------------------------------
# Build CAPEX keys (all categories + components, in encounter order)
# ---------------------------------------------------------------------------
capex_seen = {}
for sc in scenarios:
    for cat, data in sc["capex_items"].items():
        capex_seen.setdefault((cat, ""), None)
        for comp in data["components"]:
            capex_seen.setdefault((cat, comp), None)

capex_keys = list(capex_seen.keys())

# ---------------------------------------------------------------------------
# Build OPEX keys
# ---------------------------------------------------------------------------
opex_seen = {}
for sc in scenarios:
    for comp in sc["opex_items"]:
        opex_seen.setdefault(comp, None)

opex_keys = list(opex_seen.keys())

# ---------------------------------------------------------------------------
# Value accessors
# ---------------------------------------------------------------------------
def capex_val(sc, cat, comp):
    items = sc["capex_items"]
    if cat not in items:
        return 0.0
    if comp == "":
        return items[cat]["total"]
    return items[cat]["components"].get(comp, 0.0)

def opex_val(sc, comp):
    return sc["opex_items"].get(comp, 0.0)

# ---------------------------------------------------------------------------
# Build rows
# ---------------------------------------------------------------------------
n = len(scenarios)
rows = []

# Metadata
rows.append(["BESS Capacity", "MW"]  + [f"{sc['bess_mw']:.0f}"  for sc in scenarios])
rows.append(["BESS Duration", "h"]   + [str(sc["duration_h"])    for sc in scenarios])
rows.append(["BESS Energy",   "MWh"] + [f"{sc['bess_mwh']:.0f}" for sc in scenarios])

# ---------------- CAPEX ----------------
rows.append(["CAPEX", "$/kWh"] + [""] * n)

all_cats = list(dict.fromkeys(cat for cat, _ in capex_keys))

for cat in all_cats:
    rows.append(
        [cat, "$/kWh"] +
        [f"{capex_val(sc, cat, ''):.4f}" for sc in scenarios]
    )
    comps = [(c, comp) for c, comp in capex_keys if c == cat and comp != ""]
    for _, comp in comps:
        rows.append(
            [f"  {comp}", "$/kWh"] +
            [f"{capex_val(sc, cat, comp):.4f}" for sc in scenarios]
        )

rows.append(
    ["TOTAL CAPEX", "$/kWh"] +
    [f"{sc['capex_kwh']:.4f}" for sc in scenarios]
)

rows.append(["", ""] + [""] * n)

# ---------------- OPEX ----------------
rows.append(["OPEX", "$/kWh-yr"] + [""] * n)

for comp in opex_keys:
    rows.append(
        [comp, "$/kWh-yr"] +
        [f"{opex_val(sc, comp):.4f}" for sc in scenarios]
    )

rows.append(
    ["TOTAL OPEX", "$/kWh-yr"] +
    [f"{sc['opex_kwh']:.4f}" for sc in scenarios]
)

# ---------------------------------------------------------------------------
# Write CSV
# ---------------------------------------------------------------------------
OUTPUT_CSV = "standalone_bess_capex_opex_sweep.csv"

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Metric", "Unit"] + [f"Scenario_{i}" for i in range(n)])
    writer.writerows(rows)

print(f"CSV saved → {OUTPUT_CSV}")
print(f"Scenarios: {n} | CAPEX rows: {len(capex_keys)} | OPEX rows: {len(opex_keys)}")

CSV saved → standalone_bess_capex_opex_sweep.csv
Scenarios: 40 | CAPEX rows: 45 | OPEX rows: 7
